In [6]:
import numpy as np
import librosa
import os
import random
from sklearn.model_selection import train_test_split

# Constants for preprocessing
DATASET_PATH = "C:\\Users\\Chu Ting\\Desktop\\NUTN\\3-1\\SS\\Final_Project\\CREMA-D\\dataset"
DURATION = 2.5  # 2.5 seconds
SAMPLE_RATE = 22050  # 22.05 kHz
FRAME_LENGTH = 2048
HOP_LENGTH = 512
N_MFCC = 20
EMOTION_CLASSES = ['ANG', 'DIS', 'FEA', 'HAP', 'NEU', 'SAD']

# Function to preprocess audio files
def preprocess_audio(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    audio = librosa.util.fix_length(data=audio, size=int(DURATION * sr))
    return audio

# Function for data augmentation (adding noise and pitch shifting)
def augment_audio(audio):
    # Add noise
    noise = np.random.normal(0,1, audio.shape)*0.035
    noisy_audio = audio + noise

    # Shift pitch
    pitch_shifted = librosa.effects.pitch_shift(audio, sr=SAMPLE_RATE, n_steps=0.7)

    return [noisy_audio, pitch_shifted]

# Function to extract MFCC features
def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)
    return mfcc.T

# Load dataset and preprocess
def load_data():
    X, y = [], []
    for emotion_idx, emotion in enumerate(EMOTION_CLASSES):
        emotion_path = os.path.join(DATASET_PATH, emotion)
        for file in os.listdir(emotion_path):
            file_path = os.path.join(emotion_path, file)
            audio = preprocess_audio(file_path)

            # Original and augmented data
            X.append(extract_mfcc(audio))
            y.append(emotion_idx)
            
            for augmented_audio in augment_audio(audio):
                X.append(extract_mfcc(augmented_audio))
                y.append(emotion_idx)

    return np.array(X, dtype=object), np.array(y)

# Main preprocessing function
if __name__ == "__main__":
    X, y = load_data()
    X = np.array([x.flatten() for x in X], dtype=np.float32)  # Ensure float32 type
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    np.savez("4.npz", X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)
    print("Data preprocessing and feature extraction complete. Output saved to preprocessed_data.npz.")


Data preprocessing and feature extraction complete. Output saved to preprocessed_data.npz.
